In [1]:
#!pip install statsbombpy scikit-learn xgboost

In [3]:
import pandas as pd
import numpy as np
from statsbombpy import sb
import warnings
warnings.filterwarnings('ignore')

print("⏳ Pobieranie potężnej bazy danych (Cel: 15k-20k strzałów)...")
print("To może zająć kilka lub kilkanaście minut. Trwa łączenie z API StatsBomb...")

# Lista darmowych turniejów StatsBomb (Mundiale, Euro, La Liga Messi, Champions League)
tournaments = [
    (43, 106),  # World Cup 2022
    (43, 3),    # World Cup 2018
    (55, 43),   # Euro 2020
    (9, 281),   # Bundesliga 23/24 (Bayer Leverkusen)
    (11, 90),   # La Liga 2020/2021
    (11, 42),   # La Liga 2019/2020
    (11, 4),    # La Liga 2018/2019
    (11, 1),    # La Liga 2017/2018
    (11, 2),    # La Liga 2016/2017
    (16, 4),    # Champions League 2018/2019
    (16, 1),    # Champions League 2017/2018
    (16, 2),    # Champions League 2016/2017
]

shots_list = []
total_matches = 0

for comp_id, seas_id in tournaments:
    try:
        matches = sb.matches(competition_id=comp_id, season_id=seas_id)
        match_ids = matches['match_id'].tolist()
        total_matches += len(match_ids)
        print(f"Pobieranie {len(match_ids)} meczów z turnieju {comp_id}/{seas_id}...")
        
        for match_id in match_ids:
            try:
                # Ograniczamy printy, żeby nie zalać konsoli
                events = sb.events(match_id=match_id)
                if 'type' in events.columns:
                    match_shots = events[events['type'] == 'Shot'].copy()
                    shots_list.append(match_shots)
            except:
                pass # Ignorujemy błędy pojedynczych meczów
    except Exception as e:
        print(f"Pominięto turniej {comp_id}/{seas_id}: {e}")

all_shots = pd.concat(shots_list, ignore_index=True)
print("==================================================")
print(f"🎯 Sukces! Przeskanowano łącznie {total_matches} meczów.")
print(f"🔥 Zebrano potężną bazę {len(all_shots)} strzałów do treningu!")
print("==================================================")

⏳ Pobieranie potężnej bazy danych (Cel: 15k-20k strzałów)...
To może zająć kilka lub kilkanaście minut. Trwa łączenie z API StatsBomb...
Pobieranie 64 meczów z turnieju 43/106...
Pobieranie 64 meczów z turnieju 43/3...
Pobieranie 51 meczów z turnieju 55/43...
Pobieranie 34 meczów z turnieju 9/281...
Pobieranie 35 meczów z turnieju 11/90...
Pobieranie 33 meczów z turnieju 11/42...
Pobieranie 34 meczów z turnieju 11/4...
Pobieranie 36 meczów z turnieju 11/1...
Pobieranie 34 meczów z turnieju 11/2...
Pobieranie 1 meczów z turnieju 16/4...
Pobieranie 1 meczów z turnieju 16/1...
Pobieranie 1 meczów z turnieju 16/2...
🎯 Sukces! Przeskanowano łącznie 388 meczów.
🔥 Zebrano potężną bazę 9821 strzałów do treningu!


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import brier_score_loss
import joblib
import os
import ast
import json

print("⚙️ KROK 1: Feature Engineering (Tylko 7 Złotych Zasad Geometrii)...")

# 1. Odcinamy rzuty karne całkowicie! Model uczy się tylko gry z akcji i wolnych.
df = all_shots[all_shots['shot_type'] != 'Penalty'].copy()

# Wyciągamy podstawowe współrzędne
df['x'] = df['location'].apply(lambda loc: loc[0] if isinstance(loc, list) else np.nan)
df['y'] = df['location'].apply(lambda loc: loc[1] if isinstance(loc, list) else np.nan)
df = df.dropna(subset=['x', 'y']).copy()

# CECHA 1: Dystans
df['distance_to_goal'] = np.sqrt((120 - df['x'])**2 + (40 - df['y'])**2)

# CECHA 2: Kąt
def calculate_angle(x, y):
    d1 = np.sqrt((120 - x)**2 + (36 - y)**2)
    d2 = np.sqrt((120 - x)**2 + (44 - y)**2)
    if d1 * d2 == 0: return 0.0
    val = (d1**2 + d2**2 - 8**2) / (2 * d1 * d2)
    val = max(min(val, 1.0), -1.0)
    return np.degrees(np.arccos(val))

df['shot_angle'] = df.apply(lambda row: calculate_angle(row['x'], row['y']), axis=1)

# ==========================================
# 🧠 EKSTRAKCJA FUNDAMENTÓW PRZESTRZENNYCH
# ==========================================
def parse_fundamentals(row):
    frame = row.get('shot_freeze_frame', [])
    if isinstance(frame, str):
        try: frame = ast.literal_eval(frame)
        except: 
            try: frame = json.loads(frame.replace("'", '"'))
            except: frame = []
            
    if not isinstance(frame, list) or len(frame) == 0:
        return pd.Series([np.nan, np.nan, np.nan, np.nan]) 
            
    shooter_x, shooter_y = row['x'], row['y']
    gk_x, gk_y = 120.0, 40.0 
    defs_in_cone = 0
    defs_in_front = 0
    is_gk_in_cone = 0
    
    for p in frame:
        if not p.get('teammate', True):
            loc = p.get('location', [120.0, 40.0])
            dx, dy = loc[0], loc[1]
            
            # Parametry światła bramki
            t = (dx - shooter_x) / max(0.1, 120.0 - shooter_x)
            upper_y = shooter_y + t * (44.0 - shooter_y) + 1.0
            lower_y = shooter_y + t * (36.0 - shooter_y) - 1.0
            in_cone = (lower_y <= dy <= upper_y) and (dx > shooter_x)
            
            if p.get('position', {}).get('name') == 'Goalkeeper':
                gk_x, gk_y = dx, dy
                if in_cone:
                    is_gk_in_cone = 1
            else:
                if in_cone:
                    defs_in_cone += 1
                if dx >= (shooter_x - 0.5):
                    defs_in_front += 1
                            
    # CECHA 3: Odległość bramkarza od środka bramki
    gk_distance_from_line = np.sqrt((120.0 - gk_x)**2 + (40.0 - gk_y)**2)
    
    # CECHA 4 & 5: Pozycja bramkarza w stożku i liczba obrońców w stożku (Capping na max 3)
    defs_in_cone = min(defs_in_cone, 3) 
    
    # CECHA 6: Breakaway (Rajd - brak obrońców przed piłką)
    is_breakaway = 1 if defs_in_front == 0 else 0
    
    return pd.Series([gk_distance_from_line, is_gk_in_cone, defs_in_cone, is_breakaway])

print("⏳ Skanowanie geometrii boiska...")
df[['gk_distance_from_line', 'is_gk_in_cone', 'defenders_in_cone', 'is_breakaway']] = df.apply(parse_fundamentals, axis=1)

# Usuwamy strzały bez danych przestrzennych (tylko czyste dane)
df = df.dropna(subset=['defenders_in_cone']).copy()

# CECHA 7: Typ akcji i część ciała
df['is_open_play'] = (df['shot_type'] == 'Open Play').astype(int)
df['is_header'] = (df.get('shot_body_part', '') == 'Head').astype(int)

df['is_goal'] = df['shot_outcome'].apply(lambda out: 1 if out == 'Goal' else 0)

# Wektor cech
features = [
    'distance_to_goal', 
    'shot_angle', 
    'is_header', 
    'is_open_play', 
    'gk_distance_from_line', 
    'is_gk_in_cone', 
    'defenders_in_cone', 
    'is_breakaway'
]

X = df[features]
y = df['is_goal']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y) 

print("🧠 KROK 3: Trenowanie modeli zminimalizowanych do fundamentów...")

logreg = LogisticRegression(max_iter=3000, C=1.0)
logreg.fit(X_train, y_train)

rf_tuned = RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=10, random_state=42)
rf_tuned.fit(X_train, y_train)

xgb_model = XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=5, eval_metric='logloss', random_state=42)
xgb_model.fit(X_train, y_train)

print("\n✅ Wyniki Brier Score (prawdziwa siła modeli bez sztucznych karnych):")
print(f"LogReg  -> {brier_score_loss(y_test, logreg.predict_proba(X_test)[:, 1]):.4f}")
print(f"RF      -> {brier_score_loss(y_test, rf_tuned.predict_proba(X_test)[:, 1]):.4f}")
print(f"XGBoost -> {brier_score_loss(y_test, xgb_model.predict_proba(X_test)[:, 1]):.4f}")

os.makedirs('../models', exist_ok=True)
joblib.dump(logreg, '../models/xg_logreg.joblib')
joblib.dump(rf_tuned, '../models/xg_rf_tuned.joblib')
joblib.dump(xgb_model, '../models/xg_xgboost.joblib')
joblib.dump(X.columns.tolist(), '../models/xg_columns.joblib')
print("✅ Zapisano oczyszczone modele!")

⚙️ KROK 1: Feature Engineering (Tylko 7 Złotych Zasad Geometrii)...
⏳ Skanowanie geometrii boiska...
🧠 KROK 3: Trenowanie modeli zminimalizowanych do fundamentów...

✅ Wyniki Brier Score (prawdziwa siła modeli bez sztucznych karnych):
LogReg  -> 0.0809
RF      -> 0.0809
XGBoost -> 0.0815
✅ Zapisano oczyszczone modele!
